In [2]:

import torch
import pandas as pd
from pykeen.triples import TriplesFactory
from pykeen.models import DistMult, CompGCN, NodePiece
from pykeen.training import SLCWATrainingLoop
from pykeen.losses import MarginRankingLoss
from pykeen.evaluation import RankBasedEvaluator, SampledRankBasedEvaluator
from torch.optim import Adam, RMSprop, NAdam

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [3]:
main_data = pd.read_csv('../data/edges/triples.csv')
main_data = main_data.astype(str)

triples = main_data[['id_entity_1', 'predicate', 'id_entity_2']].values
triplet_data = TriplesFactory.from_labeled_triples(triples, create_inverse_triples=False)
training_set, testing_set, validation_set = triplet_data.split([0.8, 0.1, 0.1], random_state=17)


In [6]:
EMB_DIM = 128
LR = 1e-3
MARGIN = 1
WEIGHT = 1e-3
EPOCHS = 5
BATCH_SIZE = 8192
NUM_NEGS_PER_POS = 1

loss_function = MarginRankingLoss(margin=MARGIN)

model = DistMult(
    triples_factory=training_set,
    embedding_dim=EMB_DIM,
    random_seed=17,
    loss = loss_function,
)
model = model.to(device)


optimizer = Adam(params=model.parameters(), lr=LR)

training_loop = SLCWATrainingLoop(
    model=model,
    triples_factory=training_set,
    optimizer=optimizer,
    negative_sampler='basic',
    negative_sampler_kwargs=dict(
        num_negs_per_pos=NUM_NEGS_PER_POS
    )
)

training_loop.train(
    num_epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    triples_factory=training_set,
    use_tqdm_batch=False,
)

evaluator = RankBasedEvaluator()

model_results = evaluator.evaluate(
    model=model,
    mapped_triples=testing_set.mapped_triples.to(device),
    # additional_filter_triples=[
    #         training_set.mapped_triples.to(device),
    #         validation_set.mapped_triples.to(device),
    #     ],
)


metrics = model_results.to_df()
metrics = metrics[(metrics['Side'] == 'both') & (metrics['Rank_type'] == 'realistic')]
metrics

Training epochs on cuda:0: 100%|██████████| 5/5 [02:02<00:00, 24.43s/epoch, loss=1, prev_loss=1]  
The filtered setting was enabled, but there were no `additional_filter_triples`
given. This means you probably forgot to pass (at least) the training triples. Try:

    additional_filter_triples=[dataset.training.mapped_triples]

Or if you want to use the Bordes et al. (2013) approach to filtering, do:

    additional_filter_triples=[
        dataset.training.mapped_triples,
        dataset.validation.mapped_triples,
    ]

Evaluating on cuda:0:   0%|          | 760/506k [00:06<1:07:30, 125triple/s]  


KeyboardInterrupt: 

In [17]:
evaluator = RankBasedEvaluator()

model_results = evaluator.evaluate(
    model=model,
    mapped_triples=testing_set.mapped_triples[:10000].to(device),
    additional_filter_triples=[
            training_set.mapped_triples.to(device),
            validation_set.mapped_triples.to(device),
            testing_set.mapped_triples.to(device)
        ],
)


metrics = model_results.to_df()
metrics = metrics[(metrics['Side'] == 'both') & (metrics['Rank_type'] == 'realistic')]
metrics

Evaluating on cuda:0: 100%|██████████| 10.0k/10.0k [01:24<00:00, 118triple/s]


,Side,Rank_type,Metric,Value
5,both,realistic,count,2.000000e+04
14,both,realistic,inverse_geometric_mean_rank,2.521056e-06
23,both,realistic,adjusted_inverse_harmonic_mean_rank,-5.503620e-06
32,both,realistic,median_absolute_deviation,4.024902e+05
41,both,realistic,adjusted_geometric_mean_rank_index,-4.498573e-03
50,both,realistic,variance,9.683852e+10
59,both,realistic,adjusted_arithmetic_mean_rank_index,-7.137566e-03
68,both,realistic,geometric_mean_rank,3.966592e+05
77,both,realistic,arithmetic_mean_rank,5.405153e+05
86,both,realistic,standard_deviation,3.111889e+05


In [15]:
metrics = model_results.to_df()
metrics = metrics[(metrics['Side'] == 'head') & (metrics['Rank_type'] == 'realistic')]
metrics

,Side,Rank_type,Metric,Value
3,head,realistic,count,1.000000e+04
12,head,realistic,inverse_geometric_mean_rank,2.525806e-06
21,head,realistic,adjusted_inverse_harmonic_mean_rank,-5.349777e-06
30,head,realistic,median_absolute_deviation,4.022700e+05
39,head,realistic,adjusted_geometric_mean_rank_index,-2.406831e-03
48,head,realistic,variance,9.698322e+10
57,head,realistic,adjusted_arithmetic_mean_rank_index,-6.737839e-03
66,head,realistic,geometric_mean_rank,3.959132e+05
75,head,realistic,arithmetic_mean_rank,5.403963e+05
84,head,realistic,standard_deviation,3.114213e+05
